# OddsAPI Raw Game Ingestion
Ingest raw EPL game odds for spread and H2H odds.

In [0]:
# Run base file with functions and import libraries
import requests as r
import os 
import yaml
import sys
import datetime as dt
from pyspark.sql import SparkSession
import polars as pl

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Set sys path for local helper files
sys.path.append(os.getcwd())

import helpers as h

### Read in OddsAPI Credentials

In [0]:
# Load in credentials for API key
with open('/Workspace/Users/sam.ivanecky3@gmail.com/epl-bet/creds/creds.yaml') as f:
  creds = yaml.safe_load(f)

# Extract API key
api_key = creds['paid_api_key']


### Get Current Events

In [0]:
# Get current events
current_events = h.get_current_events(api_key=api_key)

In [0]:
# Get current odds
current_odds = h.get_current_odds(api_key=api_key, markets=['h2h', 'spreads'], event_ids=current_events, spark=spark)

In [0]:
# Iterate through and extract current odds for each available event
for i in current_events['id']:
    temp_odds = h.get_current_odds(api_key=api_key, markets=['h2h', 'spreads'], event_id=i)
    # Check if df exists, if not create, if so append
    try:
        current_odds = pl.concat([current_odds, temp_odds])
    except:
        current_odds = temp_odds


In [0]:
# Add field for load date of the data
# Including timestamp for intra-day odds fluctuations
current_odds = current_odds.with_columns(
    pl.lit(dt.datetime.now()).alias('load_ts')
    , pl.lit(dt.date.today().strftime('%Y-%m-%d')).alias('load_d')
)

In [0]:
spark.sql(f"DESCRIBE {FULL_TABLE_NAME}").show()

In [0]:
# Define table location
CATALOG = "nfl_bet"
SCHEMA = "raw_odds"
TABLE = "game_odds_raw"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE}"

# Convert Polars → Pandas → Spark
current_odds = current_odds.with_columns(pl.col('load_d').cast(pl.Date))
temp_pandas = current_odds.to_pandas()
spark_df = spark.createDataFrame(temp_pandas)

# Write to Delta table
spark_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(FULL_TABLE_NAME)

print(f"✓ Wrote {spark_df.count()} rows to {FULL_TABLE_NAME}")